# **Q-TRANSFER LLC**

### **Short Summary**

This notebook uses a fictional knowledge base for **Q-Transfer**, a fictional fintech company, to build and test a **Retrieval-Augmented Generation (RAG)** application.

The main purpose of this notebook is to interact with the RAG system by asking questions and retrieving relevant information from the Q-Transfer knowledge base based on **semantic similarity**. The retrieved information is then used to provide context for generating more relevant and accurate answers.

If you are viewing this notebook and would like to run the project yourself, start by checking the **`vectorizer.ipynb`** notebook. It contains the steps used to process the knowledge base, generate embeddings, and create the vector store used by this notebook.

I have also included a collection of relevant questions in the **`README.md`** file that you can use to test the RAG application's retrieval and question-answering capabilities.

In [1]:
# import the necessary library

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
import gradio as gr

c:\Users\DELL\Documents\LLM Projects\q_transfer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Constants

MODEL = "gpt-4.1-nano"
HF_DATABASE = "hf_vector_db"
OAI_DATABASE = "oai_vector_db"

In [3]:
# Load the OpenAI API KEY from .env

load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print("OpenAI Key successfully loaded.")
else:
    print("OpenAI key not loaded.")

OpenAI Key successfully loaded.


In [4]:
# Create instances of te embedders to use

hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10286.28it/s]


In [5]:
# Connect to Chroma and gain access to both HuggingFace and OpenAI vector store

hf_vector_store = Chroma(persist_directory=HF_DATABASE, embedding_function=hf_embeddings)
oai_vector_store = Chroma(persist_directory=OAI_DATABASE, embedding_function=openai_embeddings)

In [6]:
# Using hugging face retriever

# retriever = hf_vector_store.as_retriever()
retriever = oai_vector_store.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [7]:
# Retrieving relevant chunks to a question

retriever.invoke("Tell me about Q-Transfer")

[Document(id='bdff24a2-0bca-435d-8d6a-b435c171a6cc', metadata={'source': '..\\q_transfer_knowledge_base\\company\\company-overview.md', 'doc_type': 'company'}, page_content='# Q-Transfer Company Overview\n\nQ-Transfer is a fictional fintech company used for a RAG engineering project. It provides local and international money transfers and investment services through mobile applications and selected branches.\n\n## Business Profile'),
 Document(id='fe77a685-94ed-4fe2-89b8-93d6e90df12e', metadata={'doc_type': 'technology', 'source': '..\\q_transfer_knowledge_base\\technology\\system-architecture.md'}, page_content='# Technology Architecture\n\nQ-Transfer uses a service-oriented digital platform.\n\n## Major Components\n\n- Mobile applications\n- Web applications\n- API gateway\n- Customer identity service\n- Transaction service\n- Ledger service\n- Payment integration service\n- Compliance screening service\n- Notification service\n- Reporting platform\n\n## Security Principle\n\nService

In [8]:
# system prompt

SYSTEM_PROMPT_TEMPLATE = """
You are an astute and expert knowledge worker of the Q-Transfer company.
You'll be giving response to users based on their questions and added context.
If you don't know the answer, please kindly say so.
Context: {context}
"""

In [9]:
# Question and answer function for Gradio

def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [10]:
# function test

answer_question("Tell me about Q-Transfer", [])

'Q-Transfer is a fictional fintech company that specializes in local and international money transfer and investment services. It offers these services through mobile applications and a network of selected branches. The company operates a service-oriented digital platform with components such as mobile and web applications, API gateway, and various supporting services like transaction processing, compliance screening, and notifications. Q-Transfer emphasizes security principles like authenticated communication, least-privilege access, and monitoring to ensure safe and reliable financial transactions. It serves a broad range of departments including engineering, product, operations, customer support, compliance, and more, to deliver seamless financial services to its customers.'

In [11]:
# Gradio launch

gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
